# params-iterable-vs-groups — ex1: polymorphic params= signature: flat iterable vs dict-list both normalize to param_groups

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `params-iterable-vs-groups`. Running the final beacon cell reports progress against the `Config: params iterable vs groups` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: params iterable vs groups` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`params-iterable-vs-groups`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "params-iterable-vs-groups"
DD_SUBTOPIC = "Config: params iterable vs groups"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Config: params iterable vs groups — quick refresher

`torch.optim.Optimizer.__init__` accepts a single positional argument `params` whose type is polymorphic:

1. **Flat iterable of `Tensor`** — every param shares the constructor-level hyperparameters.
2. **Iterable of `dict[str, Any]`** — each dict is a parameter GROUP with its own per-group hyperparameters.

Internally the optimizer normalizes everything to `self.param_groups: list[dict]`. The flat form is just a one-group convenience.

**How the optimizer tells them apart.** It peeks at the first element: if it's a `Tensor` → flat-iterable mode; if it's a `dict` → group mode. (You'll see this in the PyTorch source: `if isinstance(param_groups[0], Tensor): param_groups = [{'params': param_groups}]`.)

**You CAN'T mix them.** A list like `[tensor_a, {'params': [tensor_b]}]` is invalid — the dispatch looks at element 0 only.

**After construction, both forms produce the same shape.** `optimizer.param_groups` is always a `list[dict]`. The flat form just has `len(param_groups) == 1`.

**Why this matters for debugging.** When you set `for group in optimizer.param_groups: ...`, the loop works either way. Code that assumed a flat list of params would silently break the moment you added a second group.

### Exercise 1 — polymorphic params= signature: flat iterable vs dict-list both normalize to param_groups

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze how `torch.optim.Optimizer` normalizes its polymorphic `params=` argument: detect whether the caller passed a flat tensor iterable or a list of param-group dicts, and produce the canonical `list[dict]` form.
> Keywords: polymorphism, param_groups, normalization, isinstance-dispatch
> ```

**KCs targeted:** `params-dispatch-tensor-vs-dict`, `param-groups-always-list-of-dict`

Implement `ex1_normalize_params(params, default_lr)`. A from-scratch reimplementation of the dispatch logic that PyTorch's `Optimizer.__init__` uses.

1. Materialize `params` to a list (it may be a generator).
2. Look at the FIRST element:
   - If it's a `torch.Tensor` → flat-iterable mode. Wrap everything in a single group: `[{'params': [...all tensors...], 'lr': default_lr}]`.
   - If it's a `dict` → group mode. Pass through, but fill in `lr=default_lr` for any group that lacks an `'lr'` key.
3. Empty list → `ValueError('optimizer got an empty parameter list')` (matches the canonical PyTorch error).
4. First element is neither tensor nor dict → `TypeError`.

Output: `list[dict]` where each dict has at least `'params'` and `'lr'`.

In [ ]:
def ex1_normalize_params(params, default_lr):
    materialized = list(params)
    if not materialized:
        raise ValueError('optimizer got an empty parameter list')
    first = materialized[0]
    if isinstance(first, t.Tensor):
        return [{'params': materialized, 'lr': default_lr}]
    if isinstance(first, dict):
        out = []
        for group in materialized:
            g = dict(group)
            if 'lr' not in g:
                g['lr'] = default_lr
            out.append(g)
        return out
    raise TypeError(
        f'params must be an iterable of Tensors or dicts, '
        f'got first element of type {type(first).__name__}'
    )


<details><summary>Solution</summary>

```python
def ex1_normalize_params(params, default_lr):
    materialized = list(params)
    if not materialized:
        raise ValueError('optimizer got an empty parameter list')
    first = materialized[0]
    if isinstance(first, t.Tensor):
        return [{'params': materialized, 'lr': default_lr}]
    if isinstance(first, dict):
        out = []
        for group in materialized:
            g = dict(group)
            if 'lr' not in g:
                g['lr'] = default_lr
            out.append(g)
        return out
    raise TypeError(
        f'params must be an iterable of Tensors or dicts, '
        f'got first element of type {type(first).__name__}'
    )
```

**`isinstance(first, t.Tensor)` is correct even for `nn.Parameter`.** `nn.Parameter` IS a `torch.Tensor` subclass — `isinstance` returns True. The dispatch works on the base class.

**Why we `dict(group)` instead of mutating in place.** The caller's dict shouldn't be mutated as a side effect of optimizer construction. `dict(group)` makes a shallow copy; adding `'lr'` to the copy doesn't affect the original. PyTorch does the same.

**This is real PyTorch source.** Read `torch/optim/optimizer.py` `Optimizer.__init__` and you'll see almost exactly this dispatch — minus the extra validation hooks (per-param NaN check, fused-vs-foreach branching, etc.). The polymorphism is intentional and stable across PyTorch versions.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()